# DC3 – Arctic Sea Ice Forecasting: Submission Workflow

This notebook guides you through the **complete submission pipeline** for the DC3
Arctic Sea Ice Forecasting Challenge:

1. **Configure** your submission metadata and paths
2. **Prepare** your model predictions  *(customisable block)*
3. **Inspect** the dataset structure
4. **Validate** conformity with the DC3 grid specification
5. **Evaluate** against observations using the full DC3 pipeline
6. **Analyse** and visualise the results
7. **Submit** to the public leaderboard

> **DC3 requires one variable:** `siconc` (sea ice area fraction, 0–1) on the
> EPSG:3413 Arctic polar stereographic grid (~3.25 km spacing) with **181 daily
> lead times** covering the 2020-11-01 → 2021-05-01 Arctic winter–spring season.

## 0. Setup

Ensure the `dc-env` conda environment is activated:

```bash
conda activate dc-env
jupyter lab notebooks/submit.ipynb
```

Or install the package:

```bash
pip install -e .
pip install "dctools @ git+https://github.com/ocean-ai-data-challenges/dc-tools.git"
```

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import copy
import json
import subprocess
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import xarray as xr

# Ensure the project root is on the Python path (works from notebooks/)
PROJECT_ROOT = Path.cwd().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

## 1. Configuration

Set the metadata and paths for your submission here. This is the **only cell
you must edit** before running the notebook end-to-end.

In [ ]:
# ── Submission metadata ──────────────────────────────────────────────
MODEL_NAME  = "MyModel"               # Short model identifier (no spaces)
TEAM_NAME   = "My Team"               # Team / author name
DESCRIPTION = "Short model description"

# ── Paths ────────────────────────────────────────────────────────────
PREDICTION_DIR = Path("/tmp/dc3_submission")   # Where your predictions live
OUTPUT_DIR     = PROJECT_ROOT / "dc3_output"   # Evaluation output directory

---

## 2. Prepare your predictions  ✉️

**This is the block you should customise.** Replace the example below with the
code that loads or generates your model's sea ice concentration forecast.

### Requirements

DC3 predictions are **sea ice concentration** (`siconc`) fields on the EPSG:3413
Arctic polar stereographic grid:

| Dimension | Range | Step | Approx. points |
|-----------|-------|------|----------------|
| x (easting) | −3 850 000 m to +3 750 000 m | 3 250 m | ~2 338 |
| y (northing) | −5 350 000 m to +5 850 000 m | 3 250 m | ~3 447 |
| time | 181 daily lead times | 1 day | 181 |

**Variable:** `siconc` (dimensionless, range 0–1).  
**Auxiliary:** 2-D `lat`/`lon` arrays (in degrees) must also be included so the
pipeline can aggregate per-bin RMSD maps.

**Accepted layouts:**
- A single `.zarr` store or `.nc` file covering the full 181-day window
- A directory of per-date `.zarr` / `.nc` files
- A glob pattern (configured in `dc3/config/dc3.yaml`)

The evaluation period is **2020-11-01 to 2021-05-01** (one 181-day seasonal forecast).

In [ ]:
# =====================================================================
# ✏️  CUSTOMISE THIS CELL
#
# Replace the sample generator with your own data-loading code.
# The result should be one or more Zarr/NetCDF files written to
# PREDICTION_DIR that cover 2020-11-01 → 2021-05-01 (181 days).
# =====================================================================

# ── DC3 grid specification ────────────────────────────────────────────
DC3_X = np.arange(-3_850_000, 3_750_001, 3_250, dtype="float32")
DC3_Y = np.arange(-5_350_000, 5_850_001, 3_250, dtype="float32")
N_LEAD_TIMES = 181

# ─── Example 1: Generate random-noise sample (for testing) ───────────
PREDICTION_DIR.mkdir(parents=True, exist_ok=True)
rng = np.random.default_rng(42)

# Build 2-D lat/lon auxiliary grids (approximate)
xx, yy  = np.meshgrid(DC3_X, DC3_Y, indexing="ij")
lat_2d  = np.clip(90.0 - np.hypot(xx, yy) / 111_320.0, -90.0, 90.0).astype("float32")
lon_2d  = (np.degrees(np.arctan2(xx, yy)) % 360).astype("float32")

siconc_data = np.clip(
    rng.random((N_LEAD_TIMES, len(DC3_X), len(DC3_Y)), dtype="float32"), 0.0, 1.0
)
times = pd.date_range("2020-11-01", periods=N_LEAD_TIMES, freq="1D")

ds = xr.Dataset(
    {
        "siconc": xr.DataArray(siconc_data, dims=["time", "x", "y"],
                               attrs={"long_name": "sea ice area fraction", "units": "1"}),
        "lat":    xr.DataArray(lat_2d,  dims=["x", "y"]),
        "lon":    xr.DataArray(lon_2d,  dims=["x", "y"]),
    },
    coords={"time": times, "x": DC3_X, "y": DC3_Y},
)
store_path = PREDICTION_DIR / "siconc_2020-2021.zarr"
ds.to_zarr(str(store_path), mode="w", consolidated=True)

print(f"Sample forecast written to: {store_path}")
print(f"  Shape: {siconc_data.shape}  (time × x × y)")

# ─── Example 2: Load from a local Zarr store ─────────────────────────
#
#   PREDICTION_DIR = Path("/data/my_model")
#   # store already contains siconc (time, x, y) + lat/lon (x, y)

# ─── Example 3: Load from NetCDF files ────────────────────────────────
#
#   import glob
#   ds = xr.open_mfdataset(sorted(glob.glob("/data/my_model/*.nc")))
#   PREDICTION_DIR.mkdir(parents=True, exist_ok=True)
#   ds.to_zarr(str(PREDICTION_DIR / "siconc_2020-2021.zarr"), mode="w")

# ─── Example 4: Download from S3 (Wasabi) ────────────────────────────
#
#   import s3fs
#   fs = s3fs.S3FileSystem(
#       key="YOUR_KEY",
#       secret="YOUR_SECRET",
#       client_kwargs={"endpoint_url": "https://s3.eu-west-2.wasabisys.com"},
#   )
#   PREDICTION_DIR.mkdir(parents=True, exist_ok=True)
#   fs.get("ppr-ocean-climat/DC3/MyModel/", str(PREDICTION_DIR), recursive=True)

## 3. Inspect the dataset

Open the forecast store and verify its structure.

In [ ]:
pred_files = sorted(PREDICTION_DIR.glob("*.zarr")) or sorted(PREDICTION_DIR.glob("*.nc"))

print(f"Found {len(pred_files)} file(s) in {PREDICTION_DIR}:")
for f in pred_files[:5]:
    print(f"  {f.name}")
if len(pred_files) > 5:
    print(f"  ... and {len(pred_files) - 5} more")

In [ ]:
f0  = pred_files[0]
ds0 = xr.open_zarr(str(f0)) if (f0.suffix == ".zarr" or f0.is_dir()) else xr.open_dataset(str(f0))

print(f"x:     {ds0.x.values[0]:.0f} m  →  {ds0.x.values[-1]:.0f} m  ({len(ds0.x)} pts)")
print(f"y:     {ds0.y.values[0]:.0f} m  →  {ds0.y.values[-1]:.0f} m  ({len(ds0.y)} pts)")
print(f"time:  {str(ds0.time.values[0])[:10]}  →  {str(ds0.time.values[-1])[:10]}  ({len(ds0.time)} timesteps)")
print(f"vars:  {list(ds0.data_vars)}")
ds0

## 4. Validate against the DC3 specification

Check grid conformity, variable presence, temporal coverage, and NaN fraction
before running the (expensive) full evaluation.

In [ ]:
# ── DC3 specification constants ───────────────────────────────────────
_DC3_X_START  = -3_850_000.0
_DC3_X_STOP   =  3_750_000.0
_DC3_Y_START  = -5_350_000.0
_DC3_Y_STOP   =  5_850_000.0
_DC3_STEP     =  3_250.0
_DC3_START    = pd.Timestamp("2020-11-01")
_DC3_END      = pd.Timestamp("2021-05-01")
_N_LEAD_TIMES = 181
_MAX_NAN_FRAC = 0.10


def validate_dc3_submission(prediction_dir: Path) -> list[str]:
    """Run DC3 conformity checks. Returns a list of issues (empty = all good)."""
    issues = []

    files = sorted(prediction_dir.glob("*.zarr")) or sorted(prediction_dir.glob("*.nc"))
    if not files:
        issues.append("❌ No .zarr or .nc files found in prediction directory.")
        return issues

    print(f"Found {len(files)} prediction file(s).")

    # Open first file for detailed checks
    f0  = files[0]
    ds0 = xr.open_zarr(str(f0)) if (f0.suffix == ".zarr" or f0.is_dir()) else xr.open_dataset(str(f0))

    # Variable
    if "siconc" not in ds0.data_vars:
        issues.append("❌ Missing variable: siconc")
    else:
        print("✅ Variable 'siconc' present.")

    # x/y dimensions
    for dim, start, stop in (("x", _DC3_X_START, _DC3_X_STOP), ("y", _DC3_Y_START, _DC3_Y_STOP)):
        if dim not in ds0.dims:
            issues.append(f"❌ Missing dimension: '{dim}'")
            continue
        v = ds0[dim].values
        if not np.isclose(v[0], start, atol=_DC3_STEP):
            issues.append(f"❌ {dim} start mismatch: expected ~{start:.0f}, got {v[0]:.0f}")
        elif not np.isclose(v[-1], stop, atol=_DC3_STEP):
            issues.append(f"❌ {dim} stop mismatch: expected ~{stop:.0f}, got {v[-1]:.0f}")
        else:
            step_actual = np.diff(v).mean()
            print(f"✅ {dim} grid: {v[0]:.0f} → {v[-1]:.0f} m  ({len(v)} pts, step ~{step_actual:.0f} m)")

    # Time coverage
    n_times = len(ds0.time)
    if n_times < _N_LEAD_TIMES:
        issues.append(f"❌ Time dimension: expected {_N_LEAD_TIMES} lead times, got {n_times}.")
    else:
        t0 = pd.Timestamp(str(ds0.time.values[0])[:10])
        t1 = pd.Timestamp(str(ds0.time.values[-1])[:10])
        print(f"✅ Time: {t0.date()} → {t1.date()}  ({n_times} lead times)")

    # Auxiliary lat/lon
    for aux in ("lat", "lon"):
        if aux not in ds0.data_vars and aux not in ds0.coords:
            issues.append(f"⚠️  Auxiliary coordinate '{aux}' not found (needed for per-bin maps).")
        else:
            print(f"✅ Auxiliary coordinate '{aux}' present.")

    # NaN fraction
    if "siconc" in ds0.data_vars:
        nan_frac = float(ds0["siconc"].isnull().mean().values)
        status   = "✅" if nan_frac <= _MAX_NAN_FRAC else "❌"
        print(f"{status} siconc NaN fraction = {nan_frac:.2%}")
        if nan_frac > _MAX_NAN_FRAC:
            issues.append(f"❌ siconc NaN fraction {nan_frac:.1%} exceeds max {_MAX_NAN_FRAC:.0%}.")

    # Value range
    if "siconc" in ds0.data_vars:
        vmin = float(ds0["siconc"].min().values)
        vmax = float(ds0["siconc"].max().values)
        if vmin < -0.01 or vmax > 1.01:
            issues.append(f"⚠️  siconc values outside [0, 1]: min={vmin:.4f}, max={vmax:.4f}")
        else:
            print(f"✅ siconc value range: [{vmin:.4f}, {vmax:.4f}] (within [0, 1])")

    ds0.close()
    return issues


issues = validate_dc3_submission(PREDICTION_DIR)

if issues:
    print(f"\n━━ {len(issues)} issue(s) found ━━")
    for issue in issues:
        print(f"  {issue}")
else:
    print("\n✅ All checks passed — your submission is DC3-compliant!")

## 5. Run the full evaluation pipeline

The DC3 evaluation compares your **sea ice concentration forecast** against three
independent reference datasets:

| Reference | Variable | Type |
|-----------|----------|------|
| **AMSR2** | `z` (siconc) | Gridded satellite SIC (primary) |
| **IABP** | `Ts` (surface temperature) | In-situ drifting buoys |
| **MODIS** | `leadmap` (lead fraction) | Satellite lead maps |

The pipeline is orchestrated by `dc3/evaluate.py`, which:
1. Downloads reference data from the Wasabi S3 bucket (cached in `dc3_output/`)
2. Interpolates predictions to the reference support
3. Computes per-lead-time RMSD metrics
4. Produces `results_<model>.json` and `results_<model>_per_bins.jsonl.gz`

> **Note:** The full 181-day evaluation processes a large volume of AMSR2 data.
> Allow ~30 GB of disk space and a stable internet connection.  
> Set `end_time: "2020-12-01"` in `dc3/config/dc3.yaml` for a quick 30-day test.

### Pointing the pipeline at your model

Add your model under `dataset_references` in `dc3/config/dc3.yaml`:

```yaml
dataset_references:
  mymodel:
    - amsr2
    - iabp
    - modis
```

The cell below patches the YAML at runtime so you can evaluate without
editing the config file manually.

In [ ]:
import yaml

config_path  = PROJECT_ROOT / "dc3" / "config" / "dc3.yaml"
with open(config_path) as f:
    dc3_config = yaml.safe_load(f)

model_name_lower = MODEL_NAME.lower()
custom_config    = copy.deepcopy(dc3_config)

# Inject the local prediction path as a new source entry
local_source = {
    "dataset":             model_name_lower,
    "observation_dataset": False,
    "ignore_geometry":     True,
    "config":              "local",
    "connection_type":     "local",
    "local_root":          str(PREDICTION_DIR.resolve()),
    "file_pattern":        "**/*.zarr",
    "keep_variables":      ["siconc", "lat", "lon", "x", "y", "time"],
    "eval_variables":      ["siconc"],
    "time_tolerance":      12,
    "metrics":             ["rmsd"],
}
# Replace existing entry for this model name (if any) or append
existing = [i for i, s in enumerate(custom_config.get("sources", [])) if s.get("dataset") == model_name_lower]
if existing:
    custom_config["sources"][existing[0]] = local_source
else:
    custom_config.setdefault("sources", []).append(local_source)

# Update dataset_references
custom_config["dataset_references"] = {model_name_lower: ["amsr2"]}

# Write custom config
custom_config_path = PROJECT_ROOT / "dc3" / "config" / f"dc3_custom_{model_name_lower}.yaml"
with open(custom_config_path, "w") as f:
    yaml.dump(custom_config, f, default_flow_style=False, sort_keys=False)

print(f"Custom config written to: {custom_config_path}")
print(f"Model: {model_name_lower}")
print(f"Prediction path: {PREDICTION_DIR.resolve()}")

In [ ]:
# Launch the evaluation pipeline
cmd = [
    sys.executable, str(PROJECT_ROOT / "dc3" / "evaluate.py"),
    "--model-name",   model_name_lower,
    "--config_name",  f"dc3_custom_{model_name_lower}",
    "--data_directory", str(OUTPUT_DIR),
]

print(f"Running: {' '.join(cmd)}")
print("Downloading references + computing metrics…\n")

result = subprocess.run(cmd, capture_output=True, text=True, cwd=str(PROJECT_ROOT))

stdout = result.stdout
print(stdout[-4000:] if len(stdout) > 4000 else stdout)
if result.returncode != 0:
    print("\n━━ STDERR ━━")
    print(result.stderr[-2000:] if len(result.stderr) > 2000 else result.stderr)
else:
    print("\n✅ Evaluation completed successfully!")

## 6. Load and inspect results

The evaluation produces a JSON results file in `dc3_output/results/`.

In [ ]:
# Find results file — own model first, then TOPAZ baseline
results_file = OUTPUT_DIR / "results" / f"results_{model_name_lower}.json"
if not results_file.exists():
    results_file = OUTPUT_DIR / "results" / "results_topaz.json"
if not results_file.exists():
    results_file = PROJECT_ROOT / "dc3" / "leaderboard_results" / "results_topaz.json"

if results_file.exists():
    with open(results_file) as f:
        data = json.load(f)
    evaluated_model = data["dataset"]
    records = data["results"][evaluated_model]
    print(f"Results loaded for model: {evaluated_model}")
    print(f"Number of evaluation records: {len(records)}")
    print(f"Reference datasets: {sorted(set(r['ref_alias'] for r in records))}")
else:
    print("No results file found. Run the evaluation (section 5) first.")
    records, evaluated_model = [], model_name_lower

In [ ]:
# Parse RMSD scores grouped by reference, variable, and lead time
scores = {}  # {ref_alias: {variable: {lead_time: value}}}

for record in records:
    ref = record["ref_alias"]
    lt  = record.get("lead_time")
    for metric in record["result"]:
        if metric["Metric"] not in ("rmse", "rmsd"):
            continue
        var = metric["Variable"]
        scores.setdefault(ref, {}).setdefault(var, {})[lt] = metric["Value"]

# Summary table
for ref_name in sorted(scores):
    print(f"\n━━ {ref_name.upper()} ━━")
    for var_name in sorted(scores[ref_name]):
        vals     = list(scores[ref_name][var_name].values())
        mean_val = np.nanmean(vals)
        print(f"  {var_name:40s}  mean RMSD = {mean_val:.4f}")

## 7. Visualise results

In [ ]:
import matplotlib.pyplot as plt

if "amsr2" in scores:
    amsr2 = scores["amsr2"]

    fig, ax = plt.subplots(figsize=(11, 4))
    for var_name in sorted(amsr2):
        lts  = sorted(amsr2[var_name].keys())
        vals = [amsr2[var_name][lt] for lt in lts]
        ax.plot(lts, vals, marker=".", linewidth=1.2, label=var_name)
    ax.set_xlabel("Lead time (days)")
    ax.set_ylabel("RMSD")
    ax.set_title(f"{evaluated_model} – Sea Ice Concentration RMSD vs AMSR2")
    ax.legend(fontsize=9)
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()
else:
    print("No AMSR2 results to plot. Run the evaluation first.")

In [ ]:
# RMSD across all reference datasets (one panel each)
if len(scores) > 1:
    refs = sorted(scores)
    fig, axes = plt.subplots(1, len(refs), figsize=(5 * len(refs), 4), squeeze=False)
    for ax, ref in zip(axes[0], refs):
        for var_name in sorted(scores[ref]):
            lts  = sorted(scores[ref][var_name].keys())
            vals = [scores[ref][var_name][lt] for lt in lts]
            ax.plot(lts, vals, marker=".", linewidth=1.2, label=var_name)
        ax.set_title(ref.upper())
        ax.set_xlabel("Lead time (days)")
        ax.set_ylabel("RMSD")
        ax.legend(fontsize=8)
        ax.grid(True, alpha=0.3)
    fig.suptitle(f"{evaluated_model} – RMSD by reference dataset")
    plt.tight_layout()
    plt.show()

## 8. Prepare your leaderboard submission

To participate in the public leaderboard, two files produced by the evaluation
are required:

| File | Content |
|------|---------|
| `results_<model>.json` | Aggregated RMSD scores per variable and lead time |
| `results_<model>_per_bins.jsonl.gz` | Per-bin spatial RMSD maps (2°×2° grid) |

In [ ]:
results_dir   = OUTPUT_DIR / "results"
json_file     = results_dir / f"results_{model_name_lower}.json"
per_bins_file = results_dir / f"results_{model_name_lower}_per_bins.jsonl.gz"

print("Files required for leaderboard submission:\n")
for path in [json_file, per_bins_file]:
    exists = "✅" if path.exists() else "❌ (not found)"
    size   = f"({path.stat().st_size / 1024:.0f} KB)" if path.exists() else ""
    print(f"  {exists}  {path.name}  {size}")

if json_file.exists() and per_bins_file.exists():
    print("\n✅ Both files are ready. Follow the steps below to submit.")
else:
    print("\n⚠️  Run the evaluation (section 5) to generate these files.")

### How to submit

1. **Open an issue** or **Pull Request** on the DC3 repository:  
   <https://github.com/ppr-ocean-ia/dc3-sea-ice-forecasting>

2. **Attach** `results_<model>.json` and `results_<model>_per_bins.jsonl.gz`.

3. Include a brief **model description**, training data summary, and an optional
   URL (paper or repository).

4. Once validated, your model will appear on the public leaderboard.

See the [submissions guide](../docs/source/content/submissions.md) for details.

## 9. Clean up (optional)

Remove the custom config and temporary prediction files to free disk space.

In [ ]:
# Uncomment the lines below to clean up temporary files
#
# custom_config_path.unlink(missing_ok=True)
# print(f"Removed custom config: {custom_config_path}")
#
# import shutil
# if PREDICTION_DIR.exists():
#     shutil.rmtree(PREDICTION_DIR)
#     print(f"Removed predictions: {PREDICTION_DIR}")